# R3-U2: AST partial fine-tuning under Protocol B
Select a Colab **T4 GPU** runtime. Sync the new code, `configs/r3_u2_ast_ft_b.json`, the R2-U2 warm-up checkpoint (`runs/r2/option_b_1b3757e898da4baaad2bde49fa3cea01/best.pt`), the frozen manifests, the R2-S1 B manifest and the cohort shards to Drive first. Run cells in order. Two experiments (baseline and one lower-backbone-lr alternative) train the last two AST blocks, the final norm and the head from the warm-up weights; each is about 45 minutes. Checkpoints and the ledger are written to Drive and a disconnected run can be resumed. No test and no Chatterbox audio is read. The recipe was agreed by the team (reported by the project owner on 12 November 2026).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import csv, hashlib, json, os, subprocess, sys, time
ROOT = Path('/content/drive/MyDrive/PAI&AIES Project')  # Change if your project is elsewhere.
SHARDS = ROOT / 'data/processed/cohort_v1'
LEDGER = ROOT / 'reports/ledger.csv'
OWNER = 'Subhan'  # Person running this notebook; written to the ledger.
MODEL_CACHE = '/content/hf_cache'  # The pinned AST is downloaded here and hash-verified; Colab-local, not synced.
# For a new run leave RUN_ID blank. To resume, enter the exact previous RUN_ID (code, environment and recipe must be unchanged).
RUN_ID = ''
RESUME = bool(RUN_ID)
if not RESUME:
    RUN_ID = 'r3_u2_' + time.strftime('%Y%m%dT%H%M%SZ', time.gmtime())
OUTPUT = ROOT / 'runs/r3' / RUN_ID
print('Run:', RUN_ID, 'Resume:', RESUME, 'Output:', OUTPUT)
assert (ROOT / 'scripts/r3_u2_ast_ft.py').is_file(), 'Sync the updated project code to Drive'
os.chdir(ROOT)
os.environ['PYTHONPATH'] = str(ROOT / 'src')
os.environ['PYTHONUNBUFFERED'] = '1'
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
sys.path.insert(0, str(ROOT / 'src'))  # lets this kernel import echoproof, not only the subprocesses

In [ ]:
import torch, transformers, numpy
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, '| torch', torch.__version__,
      '| transformers', transformers.__version__, '| numpy', numpy.__version__)
assert torch.cuda.is_available(), 'Choose a GPU runtime'
recipe = json.loads((ROOT / 'configs/r3_u2_ast_ft_b.json').read_text(encoding='utf-8'))
CHECKPOINT = ROOT / recipe['init']['checkpoint']
required = ['configs/r3_u2_ast_ft_b.json', recipe['init']['checkpoint'], 'data/manifests/cohort_v1.parquet', 'data/splits/protocol_B_v1.parquet',
            'data/processed/chunks_v1.parquet', 'data/features/r2_s1_B/r2_s1_manifest.json', 'reports/r1/split_report.json']
missing = [p for p in required if not (ROOT / p).is_file()]
assert not missing, f'Missing inputs: {missing}'
assert SHARDS.is_dir(), 'Set SHARDS to the canonical shard directory'
def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1048576), b''):
            h.update(block)
    return h.hexdigest()
assert sha256(CHECKPOINT) == recipe['init']['checkpoint_sha256'], 'The warm-up checkpoint is not the pinned best.pt'
print('Warm-up checkpoint verified:', recipe['init']['checkpoint_sha256'][:12], '| recipe status:', recipe['status'])
assert (OUTPUT / 'experiment_contract.json').is_file() if RESUME else not OUTPUT.exists(), 'Check RUN_ID and the resume setting'

## Fixture tests
Any failure or skip must be resolved before training. These tests use a tiny random AST on the GPU runtime's CPU and check the recipe, the warm-up loading refusals, the partial stage, the gradient proof, a tiny end-to-end run, the selection freeze and the export layout.

In [ ]:
tests = ['tests/test_r3_u2_ast.py', 'tests/test_r1_u2_ast.py', 'tests/test_m1_trainer.py']
check = subprocess.run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', *tests], text=True, capture_output=True)
print(check.stdout[-4000:], check.stderr[-2000:])
assert check.returncode == 0 and ' skipped' not in check.stdout, 'Resolve test failures or skips before training'

## Train, select and export
Per experiment: a gradient proof on a copy of the model, an epoch-0 evaluation that must reproduce the R2-U2 selection macro F1 (0.7913689 within 0.003), then 8 epochs. After both finish the selection is frozen and the three validation roles are scored with the winner. Resume after a disconnect by setting `RUN_ID` above and rerunning from the top.

In [ ]:
cmd = [sys.executable, 'scripts/r3_u2_ast_ft.py', '--root', str(ROOT), '--shard-dir', str(SHARDS), '--output', str(OUTPUT),
       '--ledger', str(LEDGER), '--owner', OWNER, '--model-cache', MODEL_CACHE]
if RESUME:
    cmd.append('--resume')
subprocess.run(cmd, check=True)

## Verify the handoff and write the report
Checks recorded hashes, the completed ledger rows, each run's adaptation evidence, the epoch-0 reproduction and the frozen selection. It does not replace independent review.

In [ ]:
evidence = json.loads((OUTPUT / 'evidence.json').read_text(encoding='utf-8'))
assert evidence['status'] == 'validation_predictions_complete'
assert sha256(OUTPUT / 'selection.json') == evidence['selection_sha256']
assert all(sha256(OUTPUT / name) == digest for name, digest in evidence['output_sha256'].items())
selection = json.loads((OUTPUT / 'selection.json').read_text(encoding='utf-8'))
for name, digest in selection['checkpoint_sha256'].items():
    assert sha256(OUTPUT / f'{RUN_ID}-{name}' / 'best.pt') == digest
with LEDGER.open(newline='', encoding='utf-8') as f:
    ledger_rows = list(csv.DictReader(f))
runs = {RUN_ID + '-baseline', RUN_ID + '-lower_backbone_lr'}
completed = [r for r in ledger_rows if r['run_id'] in runs and r['status'] == 'completed']
assert {r['run_id'] for r in completed} == runs, 'expected two completed ledger rows'
adaptation = {}
for name in ('baseline', 'lower_backbone_lr'):
    run_dir = OUTPUT / f'{RUN_ID}-{name}'
    ev = json.loads((run_dir / 'adaptation_evidence.json').read_text(encoding='utf-8'))
    proof = json.loads((run_dir / 'gradient_proof.json').read_text(encoding='utf-8'))
    epoch0 = json.loads((run_dir / 'epoch0_metrics.json').read_text(encoding='utf-8'))
    assert ev['problems'] == [] and ev['frozen_groups_unchanged_final_and_best'] and proof['ok'] and epoch0['within_tolerance'] is True
    assert ev['unfrozen_groups_changed_best'] == ev['trainable_groups'] and ev['epochs_run'] == 8
    adaptation[name] = {'best_epoch': ev['best_epoch'], 'best_macro_f1': ev['best_macro_f1'], 'skipped_steps': ev['skipped_steps'],
                        'epoch0_macro_f1': epoch0['metrics']['macro_f1'], 'trainable_groups': ev['trainable_groups'],
                        'f1_by_epoch': ev['selection_macro_f1_by_epoch'], 'peak_vram_mb': ev['peak_vram_mb']}
paired = evidence['paired_warmup_vs_adapted']
print('selected:', selection['selected'], '| F1 by experiment:', selection['selection_macro_f1'])
print('warm-up -> adapted (selection, all):', paired['warmup']['all']['macro_f1'], '->', paired['adapted']['all']['macro_f1'])
print('unseen XTTS panel:', paired['warmup']['unseen_xtts']['macro_f1'], '->', paired['adapted']['unseen_xtts']['macro_f1'])
print('decision changes at 0.5:', paired['decision_changes_at_0.5'])
report = {'task': 'R3-U2', 'run_id': RUN_ID, 'output': str(OUTPUT), 'evidence_sha256': sha256(OUTPUT / 'evidence.json'),
          'recipe_status': recipe['status'], 'adaptation': adaptation, 'completed_ledger_rows': completed,
          'gpu': torch.cuda.get_device_name(0), 'verification': 'artifact hashes, completed ledger rows, adaptation evidence and epoch-0 reproduction verified; independent review pending'}
report_path = ROOT / 'reports/r3' / (RUN_ID + '.json')
report_path.parent.mkdir(parents=True, exist_ok=True)
with report_path.open('x', encoding='utf-8') as f:
    json.dump(report, f, indent=2, sort_keys=True)
print('Report written:', report_path)